# Interactive Manual Alignment

Code to manually alignment files utilising an interactive overlay widget.

In [ ]:
import hyperspy.api as hs
import numpy as np
import os
from scipy.ndimage import interpolation, filters
import matplotlib.pyplot as plt
import ipywidgets as widgets
%matplotlib widget

In [ ]:
import AlignmentWidget # MAKE SURE THE WIDGET .PY FILE IS IN THE SAME FOLDER WHEN RUNNING THIS NOTEBOOK

## Code for Manual Alignment

### Manually Align the HAADF data

In [ ]:
haadfmanual = hs.load(r"load/your/path/to/haadfdata")
idpcmanual = hs.load(r"load/your/path/to/idpcdata")

In [ ]:
haadfIm = haadfmanual.split()
#iDPCIm = idpcmanual.split()

Initialise the list of arrays and shifts, and append your starting image.

In [ ]:
aligned_arrays = []
shifts=[(0,0,0)]
aligned_arrays.append(haadfIm[0])

Re-run this below cell, changing i to change each image. Options to run from pos-neg or neg-pos.

Use the sliders to update the alignment. When happy, press save to print the alignment to log, then append the file to the aligned list.

In [ ]:
i=1 # start at 1 - 0 is already appended as your start point!
aligner = AlignmentWidget.ImageAligner(aligned_arrays[i-1].data, haadfIm[i].data) # if from start
#aligner = AlignmentWidget.ImageAligner(aligned_arrays[i-1].data, haadfIm[25-i].data) # if from end of list
aligner.manual_align_with_rotation(precision=0.5, rotation_range=50)

In [ ]:
aligned_arr = np.copy(aligner.aligned)
hs_aligned = hs.signals.Signal2D(aligned_arr, metadata=haadfIm[i].metadata.as_dictionary())

In [ ]:
aligned_arrays.append(hs_aligned)
# get the shifts from object then append to shift list
final_shift_x, final_shift_y, final_rot = aligner.get_shifts()
shifts.append((final_shift_x, final_shift_y, final_rot))

Once all shifts and aligned arrays are saved, continue past here!

Check the plots, then apply the shifts to iDPC.

In [ ]:
aligned_arrays[15].plot()

In [ ]:
haadfmanual.plot()

### Apply shifts to the iDPC

If aligning the iDPC images from simultaneously acquired haadf images, run this section.

In [ ]:
#shifts.reverse() if needed
print(shifts) # check

In [ ]:
# optional - run this to apply shifts to the haadf images from pre-alignment. An excellent way to sanity check the shifts list.
shifted_imgs = []
for i in range(len(haadfmanual)):
    shifted_img = shift(haadfmanual.data[i], shift=(shifts[i][1], shifts[i][0]), mode='nearest')
    shifted_hs = hs.signals.Signal2D(shifted_img, metadata = haadfIm[i].metadata.as_dictionary())
    shifted_imgs.append(shifted_hs)

In [ ]:
# run this to apply shifts to the idpc images from pre-alignment.
shifted_imgs_idpc = []
for i in range(len(idpcmanual)):
    shifted_img = shift(idpcmanual.data[i], shift=(shifts[i][1], shifts[i][0]), mode='nearest')
    shifted_hs = hs.signals.Signal2D(shifted_img, metadata = iDPCIm[i].metadata.as_dictionary())
    shifted_imgs_idpc.append(shifted_hs)

### Stacking and saving

In [ ]:
final_shift_Stack1 = hs.stack(shifted_imgs)
final_shift_Stack2 = hs.stack(shifted_imgs_idpc)

In [ ]:
final_shift_Stack1.save(r"HAADF_aligned.hspy")
final_shift_Stack2.save(r"iDPC_aligned.hspy")